In [ ]:
import hashlib, json, os, time, urllib.request
I, SPAN, TOTAL, URL = 5, 17835223120, 107011338717, "https://huggingface.co/local-inference-lab/Qwen3.8-Flash-Next-FLARE/resolve/gb202/qwen3.8-flash-next-tp1-c16-mtp5.safetensors"
start, end = I * SPAN, min(TOTAL, (I + 1) * SPAN)  # [start, end)
path = f"/kaggle/working/flare-part-{I}"
h, done, t0, tries = hashlib.sha256(), 0, time.time(), 0
with open(path, "wb") as out:
    while done < end - start:
        tries += 1
        req = urllib.request.Request(URL, headers={"Range": f"bytes={start + done}-{end - 1}"})
        try:
            with urllib.request.urlopen(req, timeout=120) as r:
                assert r.status == 206, r.status
                while True:
                    b = r.read(16 << 20)
                    if not b:
                        break
                    out.write(b); h.update(b); done += len(b)
        except Exception as e:
            print(f"try {tries} at {done}: {e!r}", flush=True)
            if tries > 20:
                raise
            time.sleep(min(60, 2 ** tries))
        print(f"{done / 2**30:.2f} GiB in {time.time() - t0:.0f} s", flush=True)
assert done == end - start, (done, end - start)
meta = {"index": I, "start": start, "end": end, "bytes": done, "sha256": h.hexdigest(),
        "seconds": round(time.time() - t0, 1), "tries": tries}
json.dump(meta, open(f"/kaggle/working/flare-part-{I}.json", "w"))
print(meta)
